# Clase en vivo 4 · Proceso de análisis y Apache Superset · Notebook de respaldo

**La pregunta de la clase:** ¿dónde debería intervenir Mercado Sur primero para recuperar la recompra web?

Este notebook recorre **en código** lo que la presentación muestra con slides: de la pregunta al indicador, a la tabla analítica y a los gráficos, y termina exportando las tablas que alimentan el dashboard en Superset. Sirve de **respaldo** si falla Superset o si querés mostrar la Parte 1 corriendo en vivo.

Los datos de Mercado Sur son ficticios y se generan por código. Antes de cada resultado importante hay una **predicción**: anotala o decila en voz alta antes de correr la celda.

| Sección | Qué hace | Slides |
| --- | --- | --- |
| 1 a 3 | La pregunta, los tres indicadores con su definición y la tabla analítica | 8 a 11 |
| 4 a 7 | ¿Cayó la recompra web?, dónde, qué se puede afirmar y qué sí cambió | 12 a 15 |
| 8 a 10 | Variables, elegir el gráfico y los errores de visualización | 16 a 35 |
| 11 a 13 | El contrato del dashboard, las tablas para Superset y el cierre | 26 a 41 |

Corré las celdas **en orden**.

## Paso 0 · Instalación

Corré esta celda primero. Si falta alguna librería, la instala en el mismo Python que ejecuta este notebook; si ya está todo, no hace nada.

In [ ]:
# Este archivo es autocontenido: instala solo lo que le falte (se corre una sola vez)
import importlib.util
import subprocess
import sys

LIBRERIAS = ("numpy", "pandas", "matplotlib")
faltan = [m for m in LIBRERIAS if importlib.util.find_spec(m) is None]
if faltan:
    print("Instalando:", ", ".join(faltan), "(puede tardar un minuto)")
    for extra in ([], ["--user"], ["--break-system-packages"]):          # si un intento falla, prueba el siguiente
        r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *extra, *faltan], capture_output=True, text=True)
        if r.returncode == 0:
            break
    else:
        print("⚠️ No se pudo instalar solo. Corré en una celda:  %pip install " + " ".join(faltan))
        print(r.stderr[-300:])
    importlib.invalidate_caches()
faltan = [m for m in LIBRERIAS if importlib.util.find_spec(m) is None]
print("✅ Librerías listas" if not faltan else f"⚠️ Todavía faltan: {faltan}")

## 0 · Preparación

Genera las tres tablas de Mercado Sur con la semilla de siempre y arma la tabla de operaciones con la región, la categoría, el semestre y las dos facturaciones. No hace falta leerla.

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import os, sqlite3
import matplotlib.pyplot as plt
"""Datos ficticios de Mercado Sur, generados desde código (sin archivos que subir).

    from mercado_sur_datos import generar_mercado_sur
    t = generar_mercado_sur()      # t["clientes"], t["productos"], t["ventas"]

Mismo código, misma semilla, mismas tablas. Ningún dato representa una situación real.
"""
import numpy as np
import pandas as pd

CATEGORIAS = {  # categoría: (precio mínimo, precio máximo, peso en las ventas)
    "Electrónica": (150, 900, 0.15),
    "Hogar": (30, 300, 0.20),
    "Indumentaria": (20, 150, 0.20),
    "Deportes": (25, 250, 0.15),
    "Alimentos": (3, 40, 0.30),
}
REGIONES = ["Centro", "Norte", "Sur", "Litoral", "Cuyo"]
PESOS_REGION = [0.30, 0.20, 0.18, 0.17, 0.15]


def generar_mercado_sur(semilla=2025, n_clientes=1500, n_ops=12000):
    rng = np.random.default_rng(semilla)

    # ---- productos: 8 por categoría
    filas, n = [], 1
    for cat, (lo, hi, _) in CATEGORIAS.items():
        for _ in range(8):
            filas.append({"id_producto": f"P{n:03d}", "categoria": cat, "nombre": f"{cat} {n:02d}",
                          "precio_lista": round(float(rng.uniform(lo, hi)), 2)})
            n += 1
    productos = pd.DataFrame(filas)

    # ---- clientes
    clientes = pd.DataFrame({
        "id_cliente": [f"C{i:04d}" for i in range(1, n_clientes + 1)],
        "region": rng.choice(REGIONES, size=n_clientes, p=PESOS_REGION),
        "fecha_alta": pd.to_datetime("2023-01-01") + pd.to_timedelta(rng.integers(0, 900, n_clientes), unit="D"),
    })
    clientes["fecha_alta"] = clientes["fecha_alta"].dt.strftime("%Y-%m-%d")

    # ---- ventas: qué pasó (fecha, quién, por qué canal, qué producto)
    fechas = pd.to_datetime("2025-01-01") + pd.to_timedelta(rng.integers(0, 365, n_ops), unit="D")
    ventas = pd.DataFrame({
        "id_venta": [f"V{i:06d}" for i in range(1, n_ops + 1)],
        "fecha": fechas,
        "id_cliente": rng.choice(clientes["id_cliente"], size=n_ops),
        "canal": rng.choice(["web", "tienda"], size=n_ops, p=[0.55, 0.45]),
    })
    pesos_cat = np.array([v[2] for v in CATEGORIAS.values()])
    cats = np.array(list(CATEGORIAS.keys()))[rng.choice(len(CATEGORIAS), size=n_ops, p=pesos_cat / pesos_cat.sum())]
    ventas["id_producto"] = [rng.choice(productos.loc[productos["categoria"] == c, "id_producto"]) for c in cats]
    ventas = ventas.merge(productos[["id_producto", "categoria", "precio_lista"]], on="id_producto", how="left")
    ventas = ventas.merge(clientes[["id_cliente", "region"]], on="id_cliente", how="left")
    ventas["unidades"] = np.where(ventas["categoria"] == "Alimentos", rng.integers(1, 9, n_ops), rng.integers(1, 5, n_ops))
    ventas["precio_unitario"] = (ventas["precio_lista"] * rng.uniform(0.97, 1.05, n_ops)).round(2)
    ventas["descuento_pct"] = rng.choice([0.0, 0.05, 0.10, 0.15, 0.20], size=n_ops, p=[0.45, 0.15, 0.20, 0.12, 0.08])

    # ---- el patrón escondido del caso: en el 2.º semestre, un segmento cae y devuelve mucho
    s2 = ventas["fecha"].dt.month > 6
    seg = (ventas["canal"] == "web") & (ventas["region"] == "Litoral") & (ventas["categoria"] == "Electrónica") & s2
    seg_indum = (ventas["canal"] == "web") & (ventas["categoria"] == "Indumentaria") & s2
    p_comp, p_dev = np.full(n_ops, 0.88), np.full(n_ops, 0.07)
    p_can = np.full(n_ops, 0.05)
    p_comp[seg], p_dev[seg], p_can[seg] = 0.64, 0.30, 0.06
    p_comp[seg_indum], p_dev[seg_indum], p_can[seg_indum] = 0.81, 0.13, 0.06
    u = rng.random(n_ops)
    ventas["estado"] = np.where(u < p_comp, "completada", np.where(u < p_comp + p_dev, "devuelta", "cancelada"))
    ventas = ventas.loc[~(seg & (rng.random(n_ops) < 0.38))].copy()          # además, el segmento vende menos

    # ---- satisfacción (con faltantes que hay que analizar, no reemplazar)
    n_v = len(ventas)
    sat = rng.choice([1, 2, 3, 4, 5], size=n_v, p=[0.05, 0.08, 0.17, 0.35, 0.35]).astype(float)
    dev = (ventas["estado"] == "devuelta").to_numpy()
    can = (ventas["estado"] == "cancelada").to_numpy()
    sat[dev] = rng.choice([1, 2, 3], size=dev.sum(), p=[0.5, 0.3, 0.2])
    sat[rng.random(n_v) < np.where(can, 0.80, np.where(dev, 0.25, 0.12))] = np.nan
    ventas["satisfaccion"] = sat

    ventas = ventas.sort_values("fecha").reset_index(drop=True)
    ventas["id_venta"] = [f"V{i:06d}" for i in range(1, len(ventas) + 1)]
    ventas["fecha"] = ventas["fecha"].dt.strftime("%Y-%m-%d")
    ventas = ventas[["id_venta", "fecha", "id_cliente", "id_producto", "canal", "unidades",
                     "precio_unitario", "descuento_pct", "estado", "satisfaccion"]]
    return {"clientes": clientes[["id_cliente", "region", "fecha_alta"]],
            "productos": productos[["id_producto", "categoria", "nombre", "precio_lista"]],
            "ventas": ventas}


VIOLETA, VERDE, NARANJA, GRIS, ROJO = "#7C5CF6", "#10D9A0", "#F59E0B", "#A9A9B8", "#E5484D"
plt.rcParams.update({"figure.figsize": (8, 4), "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.alpha": 0.25, "font.size": 11, "axes.titleweight": "bold"})
pd.options.display.float_format = "{:,.1f}".format
pd.options.display.width = 200

def c3(x, d=1): return f"{x:.{d}f}".replace(".", ",")
def pc(x, d=1): return f"{100 * x:.{d}f}".replace(".", ",") + " %"
def miles(x): return f"{x:,.0f}".replace(",", ".")

t = generar_mercado_sur()
clientes, productos, ventas = t["clientes"], t["productos"], t["ventas"]
v = ventas.merge(productos[["id_producto", "categoria"]], on="id_producto").merge(clientes[["id_cliente", "region"]], on="id_cliente")
v["fecha"] = pd.to_datetime(v["fecha"])
v["semestre"] = np.where(v["fecha"].dt.month <= 6, "S1", "S2")
v["mes_num"] = v["fecha"].dt.month
v["facturacion_neta"] = v["unidades"] * v["precio_unitario"] * (1 - v["descuento_pct"])
v["facturacion_realizada"] = np.where(v["estado"] == "completada", v["facturacion_neta"], 0.0)
comp = v[v["estado"] == "completada"]
print(f"{miles(len(clientes))} clientes · {len(productos)} productos · {miles(len(v))} operaciones de 2025")
print("Estados:", {k: pc(x) for k, x in v["estado"].value_counts(normalize=True).items()})

## 1 · La pregunta y la decisión

Un buen análisis no empieza por el dato sino por la **decisión**: qué se haría distinto según la respuesta. Escribimos la pregunta, la audiencia y la decisión en un diccionario, el **contrato**, que vamos a completar a lo largo de la clase.

> **Predicción:** ¿qué se haría distinto si la recompra cayó en una región? ¿Y si no cayó en ninguna?

In [ ]:
contrato = {
    "audiencia": "Gerencia comercial web",
    "decision": "Elegir en qué región o categoría actuar primero para recuperar la recompra web, o no actuar si no hay nada que recuperar.",
    "periodo": "",
    "filtros": [],
    "metricas": [],
    "definiciones": {},
    "orden_lectura": [],
}
for campo, valor in contrato.items(): print(f"{campo:<14} {valor if valor else '— (se completa más adelante)'}")

## 2 · Tres indicadores, tres definiciones

La guía pide tres: **clientes activos**, **tasa de recompra** y **facturación neta**. Cada uno declara **fórmula, universo, período y estados**.

| Indicador | Fórmula | Universo | Estados |
| --- | --- | --- | --- |
| Clientes activos | Clientes con al menos una operación completada | Compradores del canal en el semestre | Completadas |
| Tasa de recompra | Personas con al menos dos operaciones completadas ÷ clientes activos | Compradores del canal en el semestre | Completadas |
| Facturación neta | Unidades × precio unitario × (1 − descuento) | Operaciones del canal en el período | Todas; la **realizada**, solo completadas |

> **Predicción:** ¿cuánto vale la tasa de recompra web del primer semestre? ¿Cuál baja más, la facturación neta o la realizada?

In [ ]:
def clientes_activos(canal, semestre):
    return comp[(comp["canal"] == canal) & (comp["semestre"] == semestre)]["id_cliente"].nunique()

def tasa_recompra(canal, semestre):
    por_cliente = comp[(comp["canal"] == canal) & (comp["semestre"] == semestre)].groupby("id_cliente").size()
    return (por_cliente >= 2).mean()

def facturacion(canal, semestre, realizada=True):
    d = v[(v["canal"] == canal) & (v["semestre"] == semestre)]
    return d["facturacion_realizada" if realizada else "facturacion_neta"].sum()

kpis = pd.DataFrame({s: {"Clientes activos": clientes_activos("web", s), "Tasa de recompra (%)": 100 * tasa_recompra("web", s),
                         "Facturación neta": facturacion("web", s, False), "Facturación realizada": facturacion("web", s, True)} for s in ("S1", "S2")})
kpis["variación (%)"] = 100 * (kpis["S2"] / kpis["S1"] - 1)
display(kpis)
print(f"La neta baja {c3(-kpis.loc['Facturación neta', 'variación (%)'])} % y la realizada, {c3(-kpis.loc['Facturación realizada', 'variación (%)'])} %: no son lo mismo.")

## 3 · La tabla analítica

Las tres tablas se combinan en una **tabla analítica** con una fila por **cliente, canal y semestre**. El **grano** es lo primero que se declara, porque define qué cuenta cada fila.

> **Predicción:** ¿cuántas filas va a tener? ¿Puede haber una fila repetida?

In [ ]:
ana = (v.groupby(["id_cliente", "region", "canal", "semestre"])
         .agg(operaciones=("id_venta", "count"), completadas=("estado", lambda s: int((s == "completada").sum())), devueltas=("estado", lambda s: int((s == "devuelta").sum())),
              canceladas=("estado", lambda s: int((s == "cancelada").sum())), facturacion_neta=("facturacion_neta", "sum"), facturacion_realizada=("facturacion_realizada", "sum"))
         .reset_index())
ana["es_recurrente"] = (ana["completadas"] >= 2).astype(int)

print(f"Filas: {miles(len(ana))}  ·  clientes distintos: {miles(ana['id_cliente'].nunique())}")
print("Controles:")
print("  filas repetidas por cliente, canal y semestre:", int(ana.duplicated(["id_cliente", "canal", "semestre"]).sum()))
print("  operaciones de la tabla analítica = operaciones de ventas:", int(ana["operaciones"].sum()) == len(v))
display(ana.head(4))

## 4 · ¿Cayó la recompra web?

La premisa de la pregunta es una **caída**. Antes de buscar dónde intervenir, la verificamos con la definición del README.

> **Predicción:** ¿cayó la recompra web del primer al segundo semestre?

In [ ]:
tabla = pd.DataFrame({canal: {s: 100 * tasa_recompra(canal, s) for s in ("S1", "S2")} for canal in ("web", "tienda")}).T
tabla["variación (puntos)"] = tabla["S2"] - tabla["S1"]
display(tabla)
web = tabla.loc["web"]
print(f"Recompra web: {c3(web['S1'])} % → {c3(web['S2'])} %  ({'+' if web['variación (puntos)'] > 0 else ''}{c3(web['variación (puntos)'])} puntos). No cayó en el total.")

## 5 · Dónde: por región y por categoría

Si el total no cayó, miramos **dónde** se mueve algo. Comparamos las regiones en la misma escala.

> **Predicción:** ¿qué región baja?

In [ ]:
def recompra_region(region, semestre): 
    por = comp[(comp["canal"] == "web") & (comp["semestre"] == semestre) & (comp["region"] == region)].groupby("id_cliente").size()
    return (por >= 2).mean(), len(por)

filas = []
for r_ in ["Norte", "Cuyo", "Centro", "Sur", "Litoral"]:
    (a, na), (b, nb) = recompra_region(r_, "S1"), recompra_region(r_, "S2")
    filas.append({"región": r_, "S1 (%)": 100 * a, "S2 (%)": 100 * b, "variación (puntos)": 100 * (b - a), "compradores S1": na, "compradores S2": nb})
reg = pd.DataFrame(filas).set_index("región"); display(reg)

orden = reg.sort_values("variación (puntos)").index
fig, ax = plt.subplots(figsize=(7, 3.6))
for i, r_ in enumerate(orden):
    a, b = reg.loc[r_, "S1 (%)"], reg.loc[r_, "S2 (%)"]
    ax.plot([a, b], [i, i], color=GRIS, lw=2.5); ax.scatter([a], [i], s=110, color=GRIS, zorder=3); ax.scatter([b], [i], s=110, color=VIOLETA if r_ != "Litoral" else ROJO, zorder=3)
    ax.text(min(a, b) - 0.5, i, c3(min(a, b)) + " %", ha="right", va="center"); ax.text(max(a, b) + 0.5, i, c3(max(a, b)) + " %", va="center")
ax.set_yticks(range(len(orden))); ax.set_yticklabels(orden); ax.set_xlim(56, 76); ax.grid(axis="y", alpha=0)
ax.set_xlabel("Recompra web (%) · gris: 1.er semestre · color: 2.º semestre"); ax.set_title("Solo Litoral baja"); plt.show()

## 6 · ¿Se puede afirmar?

Litoral baja, pero con pocos compradores. El **margen de error** de una diferencia de proporciones dice cuánto puede moverse por azar: si la diferencia es menor, **no alcanza para afirmar una caída**.

> **Predicción:** ¿alcanza una baja de seis puntos con unos 200 compradores para afirmar una caída?

In [ ]:
def diferencia(p1, n1, p2, n2):
    error = np.sqrt(p1 * (1 - p1) / n1 + p2 * (1 - p2) / n2)
    return p2 - p1, error, 1.96 * error

casos = {"Total web": (tasa_recompra("web", "S1"), clientes_activos("web", "S1"), tasa_recompra("web", "S2"), clientes_activos("web", "S2")),
         "Litoral": (reg.loc["Litoral", "S1 (%)"] / 100, int(reg.loc["Litoral", "compradores S1"]), reg.loc["Litoral", "S2 (%)"] / 100, int(reg.loc["Litoral", "compradores S2"]))}
for nombre, (p1, n1, p2, n2) in casos.items():
    dif, error, margen = diferencia(p1, n1, p2, n2)
    conclusion = "SE PUEDE afirmar un cambio" if abs(dif) > margen else "NO alcanza para afirmar un cambio"
    print(f"{nombre:<10} diferencia {c3(100 * dif)} puntos · margen de error ±{c3(100 * margen)} puntos → {conclusion}")

## 7 · Lo que sí cambió

Buscamos un cambio **con mecanismo**: miramos la facturación realizada y las devoluciones por segmento (región × categoría) del canal web.

> **Predicción:** ¿en qué segmento se concentra el problema?

In [ ]:
w = v[v["canal"] == "web"]
seg = []
for (reg_, cat), d in w.groupby(["region", "categoria"]):
    f = {s: d[(d["semestre"] == s) & (d["estado"] == "completada")]["facturacion_neta"].sum() for s in ("S1", "S2")}
    dev = {s: (d[d["semestre"] == s]["estado"] == "devuelta") for s in ("S1", "S2")}
    seg.append({"segmento": f"{reg_} × {cat}", "facturación realizada S1": f["S1"], "S2": f["S2"], "variación": f["S2"] - f["S1"],
                "devolución S1 (%)": 100 * dev["S1"].mean(), "devolución S2 (%)": 100 * dev["S2"].mean(), "operaciones S1": len(dev["S1"]), "operaciones S2": len(dev["S2"])})
seg = pd.DataFrame(seg).sort_values("variación").set_index("segmento")
display(seg.head(5))

top = seg.head(6)[::-1]
fig, ax = plt.subplots(figsize=(7.5, 3.6))
ax.barh(range(len(top)), top["variación"] / 1000, color=[ROJO if "Litoral × Electrónica" in s_ else GRIS for s_ in top.index], height=0.62)
ax.set_yticks(range(len(top))); ax.set_yticklabels(top.index); ax.set_xlabel("Variación de la facturación realizada web (miles)"); ax.grid(axis="y", alpha=0); plt.show()

le = seg.loc["Litoral × Electrónica"]
print(f"Litoral × Electrónica: devoluciones {c3(le['devolución S1 (%)'])} % → {c3(le['devolución S2 (%)'])} % · facturación realizada {miles(le['variación'])}")
s2w = w[w["semestre"] == "S2"]; con_dev = set(s2w[s2w["estado"] == "devuelta"]["id_cliente"]); por = comp[(comp["semestre"] == "S2") & (comp["canal"] == "web")].groupby("id_cliente").size()
print(f"Recompra de quienes tuvieron una devolución web en S2: {pc((por[por.index.isin(con_dev)] >= 2).mean())} · de quienes no: {pc((por[~por.index.isin(con_dev)] >= 2).mean())}")

## 8 · Variables para analizar

La **ingeniería de variables** arma, desde las columnas originales, las que el análisis necesita. Acá tres: el semestre, si la persona es recurrente y el segmento.

> **Predicción:** ¿qué columnas de la tabla analítica son variables armadas y cuáles vienen de los datos?

In [ ]:
ana["segmento"] = ana["region"] + " × " + np.where(ana["canal"] == "web", "web", "tienda")
print("semestre      → sale de la fecha: enero a junio es S1, julio a diciembre es S2")
print("es_recurrente → 1 si la persona tuvo al menos dos operaciones completadas en el semestre")
print("segmento      → región y canal juntos")
display(ana[["id_cliente", "region", "canal", "semestre", "completadas", "es_recurrente", "segmento"]].head(5))

## 9 · Elegir el gráfico

Primero la pregunta, después el gráfico: **tendencia** para ver cómo evoluciona, **comparación** para ver las partes y un **detalle auditable** para comprobar el número.

> **Predicción:** ¿qué gráfico elegirías para cada pregunta?

In [ ]:
mensual = comp[comp["canal"] == "web"].groupby("mes_num")["facturacion_neta"].sum()
fig, ejes = plt.subplots(1, 2, figsize=(11, 3.6))
ejes[0].plot(range(1, 13), mensual.values / 1000, color=VIOLETA, lw=2.8, marker="o"); ejes[0].set_ylim(0, mensual.max() / 1000 * 1.2)
ejes[0].set_xticks(range(1, 13)); ejes[0].set_title("Tendencia: facturación realizada web por mes (miles)")
r2 = reg["S2 (%)"].sort_values()
ejes[1].barh(range(5), r2.values, color=[ROJO if r_ == "Litoral" else GRIS for r_ in r2.index], height=0.62); ejes[1].set_yticks(range(5)); ejes[1].set_yticklabels(r2.index)
for i, x in enumerate(r2.values): ejes[1].text(x + 0.6, i, c3(x) + " %", va="center")
ejes[1].set_xlim(0, 85); ejes[1].set_title("Comparación: recompra web, 2.º semestre (%)"); ejes[1].grid(axis="y", alpha=0)
plt.tight_layout(); plt.show()
print("Detalle auditable: Litoral × Electrónica, web, segundo semestre")
display(w[(w["region"] == "Litoral") & (w["categoria"] == "Electrónica") & (w["semestre"] == "S2")][["id_venta", "fecha", "estado", "facturacion_neta"]].sort_values("fecha").head(8))

## 10 · Errores de visualización

Cuatro errores de la guía, cada uno con **la versión engañosa y la controlada**. Primero se predice el efecto.

> **Predicción:** si el eje arranca en 1,1 millones, ¿cuánto más baja se ve la segunda barra? ¿Cuál de los cuatro errores engaña más?

In [ ]:
f1, f2 = facturacion("web", "S1", True), facturacion("web", "S2", True)
fig, ejes = plt.subplots(1, 2, figsize=(9, 3.4))
for ax, base, titulo in [(ejes[0], 1_100_000, "Eje desde 1,1 millones"), (ejes[1], 0, "Eje desde cero")]:
    ax.bar(["1.er semestre", "2.º semestre"], [f1, f2], color=[GRIS, VIOLETA], width=0.55); ax.set_ylim(base, 1_300_000); ax.set_title(titulo); ax.grid(axis="x", alpha=0)
plt.tight_layout(); plt.show()
print(f"Eje truncado: la baja real es {c3(100 * (1 - f2 / f1))} % y con el eje desde 1,1 millones se ve {c3(100 * (1 - (f2 - 1_100_000) / (f1 - 1_100_000)), 0)} % más baja.")

prod = comp.groupby("id_producto")["facturacion_neta"].sum().sort_values(ascending=False)
print(f"Torta de {len(prod)} productos: la porción mayor es {pc(prod.max() / prod.sum())} y la mediana, {pc(prod.median() / prod.sum())}: no se puede leer. Mejor, barras con los diez principales.")

le_ops = w[(w["region"] == "Litoral") & (w["categoria"] == "Electrónica") & (w["semestre"] == "S2")]; ce_ops = w[(w["region"] == "Centro") & (w["semestre"] == "S2")]
print(f"KPI sin denominador: {int((le_ops['estado'] == 'devuelta').sum())} devoluciones en Litoral × Electrónica son {int((le_ops['estado'] == 'devuelta').sum())} de {len(le_ops)} ({pc((le_ops['estado'] == 'devuelta').mean())}); "
      f"las {int((ce_ops['estado'] == 'devuelta').sum())} de Centro son {int((ce_ops['estado'] == 'devuelta').sum())} de {len(ce_ops)} ({pc((ce_ops['estado'] == 'devuelta').mean())}).")

todas = [(v[(v['canal'] == 'web') & (v['semestre'] == s)].groupby('id_cliente').size() >= 2).mean() for s in ('S1', 'S2')]
print(f"Filtro de estado: la recompra web contando todas las operaciones es {pc(todas[0])} y {pc(todas[1])}; con solo las completadas, {pc(tasa_recompra('web', 'S1'))} y {pc(tasa_recompra('web', 'S2'))}.")

## 11 · El contrato del dashboard

Siete campos que se completan **antes** de abrir Superset: audiencia, decisión, período, filtros, métricas, definiciones y orden de lectura. La revisión **devuelve preguntas, no respuestas**.

> **Predicción:** ¿cuántas pistas va a devolver este contrato?

In [ ]:
contrato.update({
    "periodo": "Enero a diciembre de 2025, primer y segundo semestre",
    "filtros": ["semestre", "región"],
    "metricas": ["Clientes activos", "Tasa de recompra", "Facturación realizada"],
    "definiciones": {
        "Clientes activos": {"formula": "clientes con al menos una operación completada", "universo": "compradores web del semestre", "periodo": "semestre", "estados": "completadas"},
        "Tasa de recompra": {"formula": "personas con al menos dos completadas ÷ clientes activos", "universo": "compradores web del semestre", "periodo": "semestre", "estados": "completadas"},
        "Facturación realizada": {"formula": "unidades × precio unitario × (1 − descuento), solo completadas", "universo": "operaciones web del semestre", "periodo": "semestre", "estados": "completadas"},
    },
    "orden_lectura": ["Pregunta como título", "Filtros", "KPIs", "Tendencia", "Comparación", "Detalle auditable", "Nota de definiciones"],
})

def revisar_contrato(c):
    pistas = []
    for campo in ("audiencia", "decision", "periodo", "filtros", "metricas", "definiciones", "orden_lectura"):
        if not c.get(campo): pistas.append(f"Falta completar «{campo}».")
    if pistas: return pistas
    if len(c["metricas"]) != 3: pistas.append("La guía pide tres indicadores: ¿cuáles cambian la decisión y cuáles se rechazan?")
    for m in c["metricas"]:
        faltan = [k for k in ("formula", "universo", "periodo", "estados") if not c["definiciones"].get(m, {}).get(k)]
        if faltan: pistas.append(f"«{m}» no declara {', '.join(faltan)}.")
    if len(c["filtros"]) > 3: pistas.append("Son muchos filtros: ¿cuáles cambian el significado del número?")
    orden = [x.lower() for x in c["orden_lectura"]]
    pos = lambda clave: next((i for i, x in enumerate(orden) if clave in x), None)
    if pos("detalle") is not None and pos("tendencia") is not None and pos("detalle") < pos("tendencia"): pistas.append("El detalle está antes que la tendencia: ¿qué se lee primero, el contexto o el detalle?")
    if pos("nota") is None: pistas.append("¿Dónde se declaran las definiciones para que el dashboard sea auditable?")
    return pistas

pistas = revisar_contrato(contrato)
print("Pistas:", len(pistas)); [print("  →", p) for p in pistas]
if not pistas: print("✔ Sin pistas: el contrato se puede defender.")

## 12 · Las tablas para Superset

El dashboard no se construye con estas variables sueltas sino con **dos datasets**: `ventas_enriquecida` (una fila por operación) y `tabla_analitica` (una fila por cliente, canal y semestre). Los exportamos en CSV y en una base SQLite, que son los mismos archivos que reciben los equipos.

In [ ]:
os.makedirs("datos_superset", exist_ok=True)
e = v.copy(); e["mes"] = e["fecha"].dt.strftime("%Y-%m"); e["fecha"] = e["fecha"].dt.strftime("%Y-%m-%d")
e["facturacion_neta"] = e["facturacion_neta"].round(2); e["facturacion_realizada"] = e["facturacion_realizada"].round(2)
e = e[["id_venta", "fecha", "mes", "semestre", "id_cliente", "region", "id_producto", "categoria", "canal", "unidades", "precio_unitario", "descuento_pct", "estado", "satisfaccion", "facturacion_neta", "facturacion_realizada"]]
analitica = ana.drop(columns="segmento").copy(); analitica[["facturacion_neta", "facturacion_realizada"]] = analitica[["facturacion_neta", "facturacion_realizada"]].round(2)
e.to_csv("datos_superset/ventas_enriquecida.csv", index=False); analitica.to_csv("datos_superset/tabla_analitica_cliente_canal_semestre.csv", index=False)
ruta_db = "datos_superset/mercado_sur.db"
if os.path.exists(ruta_db): os.remove(ruta_db)
con = sqlite3.connect(ruta_db)
for nombre, df_ in [("clientes", clientes), ("productos", productos), ("ventas", ventas), ("ventas_enriquecida", e), ("tabla_analitica", analitica)]: df_.to_sql(nombre, con, index=False)
conteo = {n_: con.execute(f"select count(*) from {n_}").fetchone()[0] for n_ in ("clientes", "productos", "ventas", "ventas_enriquecida", "tabla_analitica")}; con.close()
print("Archivos en datos_superset/:", sorted(os.listdir("datos_superset")))
print("Filas por tabla en mercado_sur.db:", conteo)

## 13 · Cierre

**¿Dónde debería intervenir Mercado Sur primero para recuperar la recompra web?**

In [ ]:
resumen = pd.DataFrame([
    ("Recompra web, total", f"{c3(100 * tasa_recompra('web', 'S1'))} % → {c3(100 * tasa_recompra('web', 'S2'))} %", "No cayó"),
    ("Recompra web, Litoral", f"{c3(reg.loc['Litoral', 'S1 (%)'])} % → {c3(reg.loc['Litoral', 'S2 (%)'])} %", "Baja, pero dentro del margen de error: se vigila"),
    ("Devoluciones en Litoral × Electrónica", f"{c3(le['devolución S1 (%)'])} % → {c3(le['devolución S2 (%)'])} %", "Lo que sí cambió, con mecanismo"),
    ("Facturación realizada web", f"{miles(f1)} → {miles(f2)}", f"Baja {c3(100 * (1 - f2 / f1))} %"),
], columns=["indicador", "primer → segundo semestre", "qué se afirma"]).set_index("indicador")
pd.set_option("display.max_colwidth", 80); display(resumen)

**La respuesta defendible:** no hay una caída de recompra web que recuperar en el total. Lo que sí cambió es **Litoral × Electrónica**, con devoluciones de 7,5 % a 36,5 %, y la baja de recompra de Litoral es una señal débil que se vigila.

**Cuatro ideas para llevarse**

1. Un dashboard responde una decisión: no es un reporte con todo.
2. Cada KPI lleva su definición: fórmula, universo y período.
3. Contexto y tendencia primero; el detalle, para auditar.
4. Decir qué no se puede afirmar es parte de la respuesta.

**Ticket de salida** (una frase por cada uno): qué voy a repetir · qué voy a verificar · qué voy a dejar de hacer.